<a href="https://colab.research.google.com/github/andrearomano-collab/ML_oxidation_notebooks/blob/main/ML_oxidation_data_processing_and_feature_selection_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Methyl-linoleate oxidation: data processing and feature selection

This notebook implements steps 2–7 of the data-analysis workflow described in Sections 2.4 and 3.1 and Figure 2 of the manuscript *Single-platform monitoring of methyl linoleate oxidation by Vocus high-resolution chemical ionisation time-of-flight mass spectrometry*.

Starting from the extracted DHS and DSI CSV files and their `log.csv` files, it:

1. constructs time-window-averaged DHS and DSI peak tables;
2. identifies the acetone–ammonium reagent-ion signals and normalises each feature;
3. compares matched samples and blanks using one-sided Wilcoxon signed-rank tests;
4. applies Bonferroni correction and retains features at corrected *p* < 0.01;
5. subtracts the corresponding normalised blank and clips negative results to zero;
6. averages replicates at each oxidation time and fuses the DHS and DSI tables;
7. calculates Pearson correlations against four anchor features and retains associations at Bonferroni-corrected *p* < 0.01.



## 1. Input data and configuration

Expected directory structure:

```text
DATA_ROOT/
├── DHS/
│   ├── log.csv
│   └── extracted DHS CSV files
└── DSI/
    ├── log.csv
    └── extracted DSI CSV files
```

`log.csv` links each analytical file to its sample identity, oxidation time, replicate and averaging window. The DHS branch follows the final recalculated-window implementation in the source notebook: the end of the logged blank is used as a reference point, with `rp − 30` to `rp` averaged for the blank and `rp + 1` to `rp + 60` for the sample. The DSI branch uses the logged `start` and `end` values directly.

In Google Colab, mount Google Drive if required and change `DATA_ROOT` to the directory containing the `DHS` and `DSI` folders. For a cloned GitHub repository, the relative paths below can be retained.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, wilcoxon

# Optional Google Colab setup:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "Path('DATA_ROOT')" # @param {type:"string"}
OUTPUT_ROOT = DATA_ROOT / 'OUTPUT'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ALPHA = 0.01

for required_path in [DATA_ROOT / 'DHS' / 'log.csv', DATA_ROOT / 'DSI' / 'log.csv']:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {required_path}. Update DATA_ROOT before continuing."
        )

print(f'Data root: {DATA_ROOT}')
print(f'Output directory: {OUTPUT_ROOT}')

## 2. Shared functions

The functions below retain the calculations used in the starting notebook while removing duplicated exploratory cells.

- Mass-feature columns are recognised from the Tofware header pattern `'m/Q <mass>'`.
- For every analytical file, the most intense feature in `m/z` 75.5–76.5 and the most intense feature in `m/z` 133.5–134.5 are located. Their mean signals are summed to give the reagent-ion signal.
- Each time-window mean is divided by that reagent-ion signal and multiplied by (10^6), giving normalised counts per second (ncps).
- Bonferroni correction is calculated as `min(p × number_of_tests, 1)`; this is mathematically equivalent to the correction used in the source notebook.


In [ ]:
def extract_mz_from_col_name(col_name):
    """Extract the numerical m/z value from a Tofware column name."""
    mq_pos = col_name.find('m/Q ')
    if mq_pos == -1:
        return None
    try:
        return float(col_name[mq_pos + len('m/Q '):].strip("' "))
    except ValueError:
        return None


def get_most_intense_mz_in_range(df, mz_range_start, mz_range_end):
    """Return the feature with the largest whole-file mean in an m/z interval."""
    candidates = [
        col for col in df.columns
        if (mz := extract_mz_from_col_name(col)) is not None
        and mz_range_start <= mz <= mz_range_end
    ]
    if not candidates:
        return None, 0.0
    means = df[candidates].mean(axis=0)
    column = means.idxmax()
    return column, float(means[column])


def bonferroni_correct(p_values):
    """Return Bonferroni-adjusted p-values."""
    values = np.asarray(p_values, dtype=float)
    return np.minimum(values * len(values), 1.0)


def build_normalised_table(data_path, mode):
    """Build the normalised sample/blank table for DHS or DSI."""
    log = pd.read_csv(data_path / 'log.csv')
    file_cache = {}
    blank_end_by_file = {}
    rows = []

    for file_name in log['File_Name'].unique():
        df = pd.read_csv(data_path / file_name)
        file_cache[file_name] = df
        blank_rows = log[
            (log['File_Name'] == file_name) & log['sample'].str.endswith('_B')
        ]
        if blank_rows.empty:
            raise ValueError(f'No blank entry in log.csv for {file_name}')
        blank_end_by_file[file_name] = float(blank_rows.iloc[0]['end'])

    for _, log_row in log.iterrows():
        file_name = log_row['File_Name']
        df = file_cache[file_name]
        is_blank = str(log_row['sample']).endswith('_B')

        mz76, mean76 = get_most_intense_mz_in_range(df, 75.5, 76.5)
        mz134, mean134 = get_most_intense_mz_in_range(df, 133.5, 134.5)
        if mz76 is None or mz134 is None:
            raise ValueError(f'Reagent-ion peaks not found in {file_name}')
        reagent_ion_signal = mean76 + mean134
        if reagent_ion_signal == 0:
            raise ValueError(f'Zero reagent-ion signal in {file_name}')

        if mode == 'DHS':
            reference_point = blank_end_by_file[file_name]
            if is_blank:
                start_time, end_time = reference_point - 30, reference_point
            else:
                start_time, end_time = reference_point + 1, reference_point + 60
        elif mode == 'DSI':
            start_time = float(log_row['start'])
            end_time = float(log_row['end'])
        else:
            raise ValueError("mode must be 'DHS' or 'DSI'")

        period = df[df['t_elapsed_Buf'].between(start_time, end_time, inclusive='both')]
        if period.empty:
            raise ValueError(
                f'Empty averaging window for {file_name}: {start_time}–{end_time}'
            )

        mz_columns = [col for col in df.columns if col.startswith("'m/Q ")]
        means_ncps = period[mz_columns].mean(axis=0) / reagent_ion_signal * 1e6

        row = {
            'File_Name': file_name,
            'Type': 'Blank' if is_blank else 'Sample',
            'Hour': int(log_row['hour']),
            'Replicate': int(log_row['replicate']),
        }
        row.update({
            f'MZ_{extract_mz_from_col_name(col)}': value
            for col, value in means_ncps.items()
        })
        rows.append(row)

    table = pd.DataFrame(rows).sort_values(
        ['File_Name', 'Hour', 'Replicate', 'Type']
    ).reset_index(drop=True)
    return log, table


def wilcoxon_select_and_subtract(table, alpha=0.01):
    """Run the paired Wilcoxon/Bonferroni selection and blank subtraction."""
    mz_columns = [col for col in table.columns if col.startswith('MZ_')]
    samples = table[table['Type'] == 'Sample']
    blanks = table[table['Type'] == 'Blank']
    paired = samples.merge(
        blanks,
        on=['File_Name', 'Hour', 'Replicate'],
        suffixes=('_sample', '_blank'),
    )

    results = []
    for mz_col in mz_columns:
        pair = paired[[f'{mz_col}_sample', f'{mz_col}_blank']].dropna()
        try:
            statistic, p_value = wilcoxon(
                pair.iloc[:, 0],
                pair.iloc[:, 1],
                alternative='greater',
                method='auto',
            )
        except ValueError:
            statistic, p_value = np.nan, np.nan
        results.append({
            'MZ_Column': mz_col,
            'Wilcoxon_Statistic': statistic,
            'P_Value': p_value,
        })

    results = pd.DataFrame(results)
    valid = results['P_Value'].notna()
    results.loc[valid, 'P_Value_Bonferroni'] = bonferroni_correct(
        results.loc[valid, 'P_Value']
    )
    significant = results.loc[
        results['P_Value_Bonferroni'] < alpha, 'MZ_Column'
    ].tolist()

    blanks_renamed = blanks.rename(columns={
        col: f'{col}_blank' for col in mz_columns
    })
    merged = samples.drop(columns='Type').merge(
        blanks_renamed.drop(columns='Type'),
        on=['File_Name', 'Hour', 'Replicate'],
        how='inner',
    )
    sample_values = merged[mz_columns].to_numpy()
    blank_values = merged[[f'{col}_blank' for col in mz_columns]].to_numpy()
    corrected_values = np.clip(sample_values - blank_values, a_min=0, a_max=None)
    background_subtracted = pd.concat(
        [
            merged[['File_Name', 'Hour', 'Replicate']].reset_index(drop=True),
            pd.DataFrame(corrected_values, columns=mz_columns),
        ],
        axis=1,
    )
    selected = background_subtracted[
        ['File_Name', 'Hour', 'Replicate', *significant]
    ].copy()
    return results, background_subtracted, selected, significant


## 3. DHS peak table and feature selection

The DHS log contains matched sample and blank entries. As in the final table-generation code of the starting notebook, the blank endpoint defines the reference point used for the fixed DHS averaging windows. Feature selection uses a one-sided Wilcoxon signed-rank test (`sample > blank`) for every mass feature, followed by Bonferroni correction.


In [ ]:
dhs_log, dhs_normalised = build_normalised_table(DATA_ROOT / 'DHS', 'DHS')
dhs_raw_features = [col for col in dhs_normalised if col.startswith('MZ_')]

print(f'DHS log rows: {len(dhs_log)}')
print(f'DHS matched sample/blank pairs: {len(dhs_log) // 2}')
print(f'DHS initial m/z features: {len(dhs_raw_features)}')
print(dhs_normalised.iloc[:3, :8].to_string(index=False))


In [ ]:
(
    dhs_wilcoxon,
    dhs_background_subtracted,
    dhs_selected,
    dhs_selected_features,
) = wilcoxon_select_and_subtract(dhs_normalised, alpha=ALPHA)

print(f'DHS features retained at Bonferroni-corrected p < {ALPHA}: '
      f'{len(dhs_selected_features)}')
print(f'DHS selected matrix: {dhs_selected.shape[0]} rows × '
      f'{len(dhs_selected_features)} feature columns')
print(dhs_wilcoxon.sort_values('P_Value').head().to_string(index=False))

assert len(dhs_selected_features) == 304, (
    f'Expected 304 selected DHS features, obtained {len(dhs_selected_features)}.'
)
assert 'MZ_130.122467' in dhs_selected_features


## 4. DSI peak table and feature selection

The same normalisation and Wilcoxon/Bonferroni procedure is applied to DSI. In accordance with the source notebook, DSI sample and blank means use the `start` and `end` windows specified directly in the DSI log.


In [ ]:
dsi_log, dsi_normalised = build_normalised_table(DATA_ROOT / 'DSI', 'DSI')
dsi_raw_features = [col for col in dsi_normalised if col.startswith('MZ_')]

print(f'DSI log rows: {len(dsi_log)}')
print(f'DSI matched sample/blank pairs: {len(dsi_log) // 2}')
print(f'DSI initial m/z features: {len(dsi_raw_features)}')
print(dsi_normalised.iloc[:3, :8].to_string(index=False))


In [ ]:
(
    dsi_wilcoxon,
    dsi_background_subtracted,
    dsi_selected,
    dsi_selected_features,
) = wilcoxon_select_and_subtract(dsi_normalised, alpha=ALPHA)

print(f'DSI features retained at Bonferroni-corrected p < {ALPHA}: '
      f'{len(dsi_selected_features)}')
print(f'DSI selected matrix: {dsi_selected.shape[0]} rows × '
      f'{len(dsi_selected_features)} feature columns')
print(dsi_wilcoxon.sort_values('P_Value').head().to_string(index=False))

assert len(dsi_selected_features) == 474, (
    f'Expected 474 selected DSI features, obtained {len(dsi_selected_features)}.'
)
assert 'MZ_344.278442' in dsi_selected_features


## 5. Data fusion

For each sampling mode, the background-subtracted values are averaged across replicates at each oxidation time. DHS and DSI identifiers are retained by adding `_DHS` and `_DSI` suffixes before merging on `Hour`. The expected result is 12 time points and 778 feature columns (304 DHS + 474 DSI).


In [ ]:
dhs_averaged = (
    dhs_selected.groupby('Hour')[dhs_selected_features]
    .mean()
    .reset_index()
    .rename(columns={col: f'{col}_DHS' for col in dhs_selected_features})
)
dsi_averaged = (
    dsi_selected.groupby('Hour')[dsi_selected_features]
    .mean()
    .reset_index()
    .rename(columns={col: f'{col}_DSI' for col in dsi_selected_features})
)

fused_778 = dhs_averaged.merge(dsi_averaged, on='Hour', how='inner')
fused_feature_columns = [col for col in fused_778 if col != 'Hour']

print(f'DHS averaged rows: {len(dhs_averaged)}')
print(f'DSI averaged rows: {len(dsi_averaged)}')
print(f'Fused table: {len(fused_778)} rows × {len(fused_feature_columns)} feature columns')
print(fused_778.iloc[:5, :8].to_string(index=False))

assert len(fused_778) == 12
assert len(fused_feature_columns) == 778


## 6. Anchor-centred Pearson correlation analysis

The four anchor features are those used in the starting notebook and Section 3.1:

| Putative assignment | Feature |
|---|---|
| Methyl-linoleate monohydroperoxide (HPOME) | `MZ_344.278442_DSI` |
| Methyl hydroxyoctadecadienoate (methyl-HODE) | `MZ_328.283234_DSI` |
| Methyl 9-oxononanoate | `MZ_204.158478_DSI` |
| 2-Heptenal | `MZ_130.122467_DHS` |

For each anchor, Pearson correlations are calculated against all 778 fused features. Bonferroni correction is applied separately within each anchor family, and associations with corrected *p* < 0.01 are retained. The source notebook includes each anchor’s self-correlation; this behaviour is preserved because it contributes one row to each family and is required to reproduce the reported total of 307.


In [ ]:
anchor_features = [
    'MZ_344.278442_DSI',
    'MZ_328.283234_DSI',
    'MZ_204.158478_DSI',
    'MZ_130.122467_DHS',
]

missing_anchors = [anchor for anchor in anchor_features if anchor not in fused_778]
if missing_anchors:
    raise KeyError(f'Anchor features missing from fused dataset: {missing_anchors}')

association_tables = []
family_counts = {}

for anchor in anchor_features:
    rows = []
    for feature in fused_feature_columns:
        if feature == anchor:
            correlation, p_value = 1.0, 0.0
        else:
            pair = fused_778[[anchor, feature]].dropna()
            correlation, p_value = pearsonr(pair[anchor], pair[feature])

        rows.append({
            'Reference Variable': anchor,
            'Target Variable': feature,
            'Pearson r': correlation,
            'P_Value': p_value,
        })

    family = pd.DataFrame(rows)
    family['P_Value_Bonferroni'] = bonferroni_correct(family['P_Value'])
    family = family[family['P_Value_Bonferroni'] < ALPHA].copy()
    family['Target Variable Mean Intensity (ncps)'] = family['Target Variable'].map(
        fused_778[fused_feature_columns].mean(axis=0)
    )
    family_counts[anchor] = len(family)
    association_tables.append(family)

correlation_associations_307 = pd.concat(association_tables, ignore_index=True)

print('Significant associations by anchor:')
for anchor, count in family_counts.items():
    print(f'  {anchor}: {count}')
print(f'Total significant anchor–feature associations: '
      f'{len(correlation_associations_307)}')
print(correlation_associations_307.head(10).to_string(index=False))

assert list(family_counts.values()) == [29, 62, 100, 116]
assert len(correlation_associations_307) == 307


## 7. Export and workflow reconciliation

The principal endpoint matching the starting notebook is the 307-row table of significant anchor–feature associations. For auditability, the notebook also reports and exports a deduplicated feature matrix: the same target feature can occur in more than one anchor family, so the number of unique target features is smaller than the number of association rows. This check does not introduce a new statistical filter.


In [ ]:
unique_correlated_features = (
    correlation_associations_307['Target Variable'].drop_duplicates().tolist()
)
correlated_features_unique_matrix = fused_778[
    ['Hour', *unique_correlated_features]
].copy()

reconciliation = pd.DataFrame([
    {'Stage': 'Initial DHS m/z features', 'Manuscript/Figure 2': 1087,
     'Observed': len(dhs_raw_features)},
    {'Stage': 'Initial DSI m/z features', 'Manuscript/Figure 2': 1571,
     'Observed': len(dsi_raw_features)},
    {'Stage': 'DHS after Wilcoxon/Bonferroni', 'Manuscript/Figure 2': 304,
     'Observed': len(dhs_selected_features)},
    {'Stage': 'DSI after Wilcoxon/Bonferroni', 'Manuscript/Figure 2': 474,
     'Observed': len(dsi_selected_features)},
    {'Stage': 'Fused DHS + DSI feature columns', 'Manuscript/Figure 2': 778,
     'Observed': len(fused_feature_columns)},
    {'Stage': 'Significant anchor–feature association rows',
     'Manuscript/Figure 2': 307, 'Observed': len(correlation_associations_307)},
    {'Stage': 'Unique correlated target features',
     'Manuscript/Figure 2': np.nan, 'Observed': len(unique_correlated_features)},
])
reconciliation['Match'] = (
    reconciliation['Manuscript/Figure 2'].isna()
    | (reconciliation['Manuscript/Figure 2'] == reconciliation['Observed'])
)

outputs = {
    'DHS_normalised_sample_blank.csv': dhs_normalised,
    'DSI_normalised_sample_blank.csv': dsi_normalised,
    'DHS_wilcoxon_results.csv': dhs_wilcoxon,
    'DSI_wilcoxon_results.csv': dsi_wilcoxon,
    'DHS_selected_304.csv': dhs_selected,
    'DSI_selected_474.csv': dsi_selected,
    'DHS_DSI_fused_778.csv': fused_778,
    'correlation_associations_307.csv': correlation_associations_307,
    'correlated_features_unique_matrix.csv': correlated_features_unique_matrix,
    'workflow_reconciliation.csv': reconciliation,
}
for file_name, dataframe in outputs.items():
    dataframe.to_csv(OUTPUT_ROOT / file_name, index=False)

print(reconciliation.to_string(index=False))
print(f'\nUnique target features represented by the 307 associations: '
      f'{len(unique_correlated_features)}')
print(f'Duplicate cross-family association rows: '
      f'{len(correlation_associations_307) - len(unique_correlated_features)}')
print(f'\nSaved {len(outputs)} output tables in: {OUTPUT_ROOT}')
print('\nUnique-feature matrix preview:')
print(correlated_features_unique_matrix.iloc[:5, :8].to_string(index=False))


## Verification result for the supplied data package

Execution with the supplied files reproduced the manuscript’s selection and fusion counts: **304 DHS**, **474 DSI**, **778 fused features**, and **307 significant anchor–feature associations** (29, 62, 100 and 116 for the four anchors).

Two discrepancies require attention before publication:

1. The supplied extracted CSV files and the original notebook generate **1,087 DHS** and **1,571 DSI** initial mass-feature columns, rather than 1,057 and 1,570 as stated in Figure 2 and Section 3.1. No undocumented feature deletion has been introduced here to force the manuscript values.
2. The 307 endpoint comprises **307 anchor–feature association rows but 213 unique target features**; 94 rows repeat features that occur in more than one anchor family. The source notebook’s total of 307 is the sum 29 + 62 + 100 + 116, rather than a deduplicated feature count.
